<a href="https://colab.research.google.com/github/Viclant/AA/blob/pagerank-lab1-results/_pagerank_lab1_student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PageRank: a guided Python and algorithms refresher

**Two approaches to ranking web pages:** a random-surfer simulation and a matrix-based power iteration.

This notebook is designed for intermediate students who know some Python and want to practise turning an algorithmic idea into a modular program.

## What you will practise

- Organising information with lists, tuples, sets, dictionaries, functions, and one small class.
- Representing a website network as a directed graph with **NetworkX**.
- Reading and writing NumPy arrays, including matrices and vectors.
- Using probability and pseudo-random numbers carefully.
- Implementing PageRank in two different ways and checking whether the answers agree.
- Measuring running time as the number of pages and links changes.
- Reading interactive graph and chart visualisations.

## How to use this notebook

Run the cells from top to bottom. Complete the coding tasks marked **Your turn** and fill in every `TODO` in the code cells. The main programming tasks are to write a reverse-link lookup helper, implement the random-surfer algorithm, construct the link-following and Google matrices, and implement power iteration. Later cells depend on these functions, so complete them before continuing.

The graph and chart visualisation helpers, along with the cells that call them, are provided as working code. You do not need to implement or modify visualisation code.

The examples use short website-style names such as `search.com`, `social.com`, and `shop.com`. The graph is deliberately small at first so that we can inspect it.

**Libraries:** Python's standard library, NumPy, pandas, NetworkX, Matplotlib, and Plotly. Plotly figures are interactive: hover over points, zoom, and pan. The graph positions are computed by NetworkX.

## 0. Setup: imports and notebook tools

We will use:

- `networkx` to store directed graphs and calculate graph layouts.
- `numpy` for vectors, matrices, and fast matrix-vector multiplication.
- `pandas` to display tables with meaningful row and column labels.
- `plotly` for interactive graph, matrix, and performance visualisations.
- `random` and `time` from the Python standard library for simulation and timing.

If a library is missing in your environment, install it once in a terminal or notebook cell with `%pip install networkx numpy pandas matplotlib plotly scipy`.

In [ ]:
import random
import time
import platform
from statistics import median

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from IPython.display import display

print("Python:", platform.python_version())
print("NumPy:", np.__version__)
print("NetworkX:", nx.__version__)
print("pandas:", pd.__version__)


Python: 3.10.6
NumPy: 1.26.2
NetworkX: 3.3
pandas: 2.2.2


## 1. The problem: a web of pages and links

A web graph is a **directed graph**:

- Each page is a node (also called a vertex).
- A link from one page to another is a directed edge.
- `search.com -> social.com` means that `search.com` contains a link to `social.com`.

A page can have many incoming links, many outgoing links, or no outgoing links. A page with no outgoing links is sometimes called a *dangling page*.

PageRank gives each page a score. A page tends to receive a higher score when important pages link to it. The score is not simply the number of incoming links: the importance of the pages providing those links matters too.

We will use a small illustrative network. `weather.com` is included as a dangling page so that we can practise handling that case.

### 1.1 A quick refresher: Python data structures

We will represent pages and links with ordinary Python objects before using NetworkX.

- A **list** stores an ordered collection and can be changed: `pages[0]`, `pages.append(...)`.
- A **tuple** is an ordered collection that is usually used as a fixed record. We will use `(source, target)` for a link.
- A **set** stores unique values. It is useful for removing duplicates and checking membership.
- A **dictionary** maps keys to values. We will use it to map each page to its outgoing links, and later to map each page to its PageRank score.

These structures solve different problems. Choose the structure that makes the operation clear.

In [2]:
# A list: order is meaningful here because we will use it as a matrix order later.
pages = ["search.com", "social.com", "shop.com", "news.com", "weather.com"]

# A list of tuples: each tuple is one directed link (source, destination).
links = [
    ("search.com", "social.com"),
    ("search.com", "news.com"),
    ("social.com", "search.com"),
    ("social.com", "shop.com"),
    ("shop.com", "search.com"),
    ("news.com", "social.com"),
]

# A set keeps only unique page names.
unique_pages = set(pages + ["search.com", "social.com"])

# A dictionary maps each page to a list of its outgoing neighbours.
outgoing_links = {page: [] for page in pages}
for source, target in links:
    outgoing_links[source].append(target)

print("Pages:", pages)
print("Number of unique pages:", len(unique_pages))
print("Outgoing links from each page:")
for page, destinations in outgoing_links.items():
    print(f"  {page:12s} -> {destinations}")


Pages: ['search.com', 'social.com', 'shop.com', 'news.com', 'weather.com']
Number of unique pages: 5
Outgoing links from each page:
  search.com   -> ['social.com', 'news.com']
  social.com   -> ['search.com', 'shop.com']
  shop.com     -> ['search.com']
  news.com     -> ['social.com']
  weather.com  -> []


**Notice:** `weather.com` has an empty list of outgoing links. We have not lost it: it is still a page in the dictionary and in the page list.

A dictionary comprehension such as `{page: [] for page in pages}` is a compact way to create one entry for each page. The loop then fills the relevant lists.

### 1.2 Functions: give one job a clear name

A function should perform one reasonably small task. This makes the program easier to test, explain, reuse, and change.

Here is a small example. It counts outgoing links using the dictionary we just built.

In [3]:
def count_outgoing_links(outgoing, page):
    """Return the number of outgoing links recorded for one page."""
    return len(outgoing[page])

for page in pages:
    print(page, "has", count_outgoing_links(outgoing_links, page), "outgoing link(s)")


search.com has 2 outgoing link(s)
social.com has 2 outgoing link(s)
shop.com has 1 outgoing link(s)
news.com has 1 outgoing link(s)
weather.com has 0 outgoing link(s)


**Your turn — implement the lookup function**

Write `pages_linking_to(outgoing, target)` so that it returns a list of every source page whose outgoing-link list contains `target`. Implement the function as a whole; the code cell provides the function signature and a test call.

In [4]:
def pages_linking_to(outgoing, target):
    result = []
    for source, destinations in outgoing.items():
        if target in destinations:
            result.append(source)
    return result

print("Pages linking to social.com:", pages_linking_to(outgoing_links, "social.com"))

Pages linking to social.com: ['search.com', 'news.com']


### 1.3 A small class: keep related data and operations together

A **class** describes a kind of object. An **instance** is one particular object created from that class.

Our `WebsiteNetwork` class stores a NetworkX graph in the attribute `graph`. Its methods add pages and links and return simple summaries. We are not building a large software framework; this small class is only a way to practise the basic idea of grouping related data and operations.

NetworkX itself already provides many graph operations. We keep the class deliberately thin.

In [5]:
class WebsiteNetwork:
    """A small wrapper around a directed NetworkX graph of web pages."""

    def __init__(self, pages=None, links=None):
        self.graph = nx.DiGraph()
        if pages is not None:
            self.graph.add_nodes_from(pages)
        if links is not None:
            self.graph.add_edges_from(links)

    def add_page(self, page):
        """Add a page. NetworkX ignores a page that is already present."""
        self.graph.add_node(page)

    def add_link(self, source, target):
        """Add a directed link; NetworkX adds missing endpoint nodes too."""
        self.graph.add_edge(source, target)

    def page_names(self):
        """Return the page names in the graph's current node order."""
        return list(self.graph.nodes())

    def number_of_links(self):
        """Return the number of directed links."""
        return self.graph.number_of_edges()


In [6]:
# Create one WebsiteNetwork instance from our lists.
website = WebsiteNetwork(pages, links)

print("Pages:", website.page_names())
print("Number of directed links:", website.number_of_links())
print("NetworkX graph type:", type(website.graph))


Pages: ['search.com', 'social.com', 'shop.com', 'news.com', 'weather.com']
Number of directed links: 6
NetworkX graph type: <class 'networkx.classes.digraph.DiGraph'>


## 2. Visualising the network with NetworkX and Plotly

A graph plot often reveals structure that is difficult to see in a list of edges. We will use `nx.spring_layout` to calculate positions. The layout is only a visual arrangement; it does **not** change the graph or its PageRank values.

The helper below uses NetworkX for the graph and layout, then Plotly for an interactive view. Hover over a node to see its name, outgoing-link count, and (when supplied) score. The arrows show link direction.

In [7]:
#@title Visualisation Helpers
def plot_web_graph(G, title="Website network", scores=None, seed=7):
    """Draw a directed NetworkX graph as an interactive Plotly figure."""
    if G.number_of_nodes() == 0:
        print("The graph is empty.")
        return

    positions = nx.spring_layout(G, seed=seed)

    # One line trace can contain all edge segments.
    edge_x = []
    edge_y = []
    for source, target in G.edges():
        x0, y0 = positions[source]
        x1, y1 = positions[target]
        edge_x.extend([x0, x1, None])
        edge_y.extend([y0, y1, None])

    edge_trace = go.Scatter(
        x=edge_x,
        y=edge_y,
        mode="lines",
        line=dict(width=1.4),
        hoverinfo="skip",
        name="links",
    )

    node_x = []
    node_y = []
    hover_text = []
    node_sizes = []
    for node in G.nodes():
        x, y = positions[node]
        node_x.append(x)
        node_y.append(y)
        score_text = "not calculated" if scores is None else f"{scores.get(node, 0):.5f}"
        hover_text.append(
            f"<b>{node}</b><br>Outgoing links: {G.out_degree(node)}<br>PageRank: {score_text}"
        )
        if scores is None:
            node_sizes.append(22)
        else:
            largest = max(scores.values()) if scores else 1
            node_sizes.append(18 + 38 * scores.get(node, 0) / largest if largest > 0 else 18)

    node_trace = go.Scatter(
        x=node_x,
        y=node_y,
        mode="markers+text",
        text=list(G.nodes()),
        textposition="top center",
        hovertext=hover_text,
        hoverinfo="text",
        marker=dict(size=node_sizes, line=dict(width=1)),
        name="pages",
    )

    # Plotly annotations draw arrowheads at the destination end of each link.
    arrows = []
    for source, target in G.edges():
        x0, y0 = positions[source]
        x1, y1 = positions[target]
        arrows.append(dict(
            x=x1, y=y1, ax=x0, ay=y0, xref="x", yref="y",
            axref="x", ayref="y", showarrow=True, arrowhead=3,
            arrowsize=1, arrowwidth=1, opacity=0.55,
        ))

    fig = go.Figure(data=[edge_trace, node_trace])
    fig.update_layout(
        title=title,
        showlegend=False,
        hovermode="closest",
        annotations=arrows,
        margin=dict(l=20, r=20, t=55, b=20),
        height=520,
        xaxis=dict(visible=False),
        yaxis=dict(visible=False),
    )
    fig.show()
    #return fig


def plot_rank_bars(scores, title="PageRank scores"):
    """Show scores in descending order as an interactive bar chart."""
    ordered = sorted(scores.items(), key=lambda item: item[1], reverse=True)
    names = [item[0] for item in ordered]
    values = [item[1] for item in ordered]
    fig = go.Figure(go.Bar(x=names, y=values, hovertemplate="%{x}<br>Score: %{y:.6f}<extra></extra>"))
    fig.update_layout(title=title, xaxis_title="Website", yaxis_title="Score", height=400)
    fig.show()
    #return fig


def plot_matrix_heatmap(matrix, labels, title="Matrix"):
    """Display a labelled matrix as an interactive heatmap."""
    fig = go.Figure(go.Heatmap(
        z=matrix,
        x=labels,
        y=labels,
        colorscale="Blues",
        hovertemplate="Row: %{y}<br>Column: %{x}<br>Value: %{z:.4f}<extra></extra>",
    ))
    fig.update_layout(title=title, xaxis_title="Source page (column)", yaxis_title="Destination page (row)", height=500)
    fig.show()
    #return fig


In [ ]:
plot_web_graph(website.graph, title="Our first website network")


### 2.1 Read the picture

The arrow `A -> B` means that page A points to page B. In a directed graph, the direction matters: `A -> B` does not automatically mean that `B -> A` exists.

Try to answer these questions before running more code:

1. Which page has the largest number of outgoing links?
2. Which page is dangling?
3. Does the page with the most outgoing links necessarily have the highest PageRank?

You can inspect the graph with `website.graph.in_degree(page)` and `website.graph.out_degree(page)`.

1. `search.com` and `social.com` tie. Each has two links going out.
2. `weather.com` is the dangling page. It has no outgoing links.
3. No. A page gains score from the pages that link to it. For example, `shop.com` sends its whole link-following share to `search.com` because that is its only link.

In [9]:
degree_summary = pd.DataFrame({
    "incoming": dict(website.graph.in_degree()),
    "outgoing": dict(website.graph.out_degree()),
})
display(degree_summary)


,incoming,outgoing
search.com,2,2
social.com,2,2
shop.com,1,1
news.com,1,1
weather.com,0,0


## 3. Probability refresher before simulating a surfer

The first PageRank method will simulate a person browsing the web. We need a few probability ideas first.

### 3.1 A probability is a number from 0 to 1

A probability of `0.2` means a 20% chance for an event in one trial. A probability of `1` means the event always happens; `0` means it never happens.

A **Bernoulli trial** has two possible outcomes: success or failure. One simple way to simulate it is:

```python
if rng.random() < probability:
    # success
else:
    # failure
```

`rng.random()` returns a pseudo-random floating-point number in the interval from 0 (inclusive) to 1 (exclusive). If the threshold is `0.2`, about 20% of many trials should satisfy the condition. The exact number in a small experiment will vary.

### 3.2 Random choice from a finite set

`rng.choice(items)` selects one item uniformly at random: each item has the same chance. In our simulation, this will select a starting page, a link to follow, or a page to teleport to.

### 3.3 Seeds and reproducibility

A pseudo-random number generator produces a deterministic sequence from its seed. The seed does **not** make the result more random; it makes an experiment repeatable. Use a local generator such as `random.Random(12)` instead of changing global random state.

For NumPy, use `np.random.default_rng(12)`. It provides methods such as `.random()`, `.integers()`, and `.choice()`.

### 3.4 Estimate a probability by repeated trials

In [ ]:
probability = 0.20
number_of_trials = 500000
rng = random.Random(2)

outcomes = [rng.random() < probability for _ in range(number_of_trials)]
observed_probability = sum(outcomes) / number_of_trials

print("Target probability:  ", probability)
print("Observed proportion: ", round(observed_probability, 4))

fig = go.Figure(go.Bar(
    x=["Success", "Failure"],
    y=[sum(outcomes), number_of_trials - sum(outcomes)],
    hovertemplate="%{x}: %{y}<extra></extra>",
))
fig.update_layout(title="A Bernoulli experiment", yaxis_title="Number of trials", height=350)
fig.show()


Target probability:   0.2
Observed proportion:  0.1995


The observed proportion is usually close to 0.20 when there are many trials, but it will not normally be exactly 0.20. This difference is **sampling variation**.

For the random-surfer method, we will make many transitions and average the visit frequencies from several independent walks. We will not discard an initial section of a walk. Instead, we will make the estimate more stable by repeating the experiment and averaging the results.

## 4. Method 1 — PageRank by random surfing

Imagine a visitor currently on one page. At each step:

1. With probability $\alpha$, the visitor tries to follow one of the current page's outgoing links, chosen uniformly.
2. With probability $1-\alpha$, the visitor teleports to a uniformly chosen page.
3. If the current page has no outgoing links, the visitor moves to a uniformly chosen page. This is our rule for a dangling page.

We use $\alpha=0.85$ as a familiar example. It means an 85% chance to follow a link and a 15% chance to teleport. It is a modelling choice, not a law of the web.

A visit is counted each time the visitor is on a page. The empirical PageRank is the fraction of counted visits belonging to each page.

To reduce the variability of one simulation, we run several independent walks. Each walk starts from a random page. We calculate the visit-frequency dictionary for each walk, then average the dictionaries. Every walk contributes equally to the final average.

### 4.1 The algorithm in plain language

For each repetition:

- choose a starting page;
- repeat `steps_per_walk` times:
  - count the current page as a visit;
  - decide whether to follow a link or teleport;
  - move to the selected next page;
- divide each page's visit count by the number of steps.

At the end, average each page's frequency over all repetitions. The result should sum to approximately 1 (in fact, to 1 up to floating-point rounding).

**Important parameters**

- `alpha`: probability of following a link when the current page has outgoing links.
- `steps_per_walk`: number of visits counted in each independent walk.
- `repeats`: number of independent walks to average.
- `seed`: optional seed for reproducible experiments.

### 4.2 Your turn — implement the random-surfer algorithm

Implement `pagerank_random_surfer(G, alpha, steps_per_walk, repeats, seed)` in the next code cell. The scaffold handles parameter checks, the empty graph, the random-number generator, and each page's outgoing-neighbour list. You are responsible for the simulation and all PageRank calculations.

For each independent walk, choose a starting page uniformly at random. At every step, count a visit to the current page, then move according to the model: a dangling page sends the surfer to a uniformly selected page; otherwise follow an outgoing link with probability `alpha`, or teleport uniformly with probability `1 - alpha`. Convert each walk's visit counts to frequencies, average those frequencies across repetitions, and return a dictionary keyed by page name.

Use the supplied `rng` for every random choice so that `seed` makes the simulation reproducible.

In [11]:
def pagerank_random_surfer(G, alpha=0.85, steps_per_walk=5000, repeats=10, seed=None):
    """Estimate PageRank by averaging visit frequencies from independent walks."""
    if not 0 <= alpha <= 1:
        raise ValueError("alpha must be between 0 and 1")
    if steps_per_walk <= 0 or repeats <= 0:
        raise ValueError("steps_per_walk and repeats must be positive")

    nodes = list(G.nodes())
    if not nodes:
        return {}

    rng = random.Random(seed)
    outgoing = {node: list(G.successors(node)) for node in nodes}
    total_scores = {node: 0.0 for node in nodes}

    for _ in range(repeats):
        visits = {node: 0 for node in nodes}
        current = rng.choice(nodes)

        for _ in range(steps_per_walk):
            visits[current] += 1
            neighbors = outgoing[current]

            if neighbors and rng.random() < alpha:
                current = rng.choice(neighbors)
            else:
                current = rng.choice(nodes)

        for node in nodes:
            total_scores[node] += visits[node] / steps_per_walk

    return {node: total_scores[node] / repeats for node in nodes}

In [ ]:
random_scores = pagerank_random_surfer(
    website.graph,
    alpha=0.85,
    steps_per_walk=12000,
    repeats=12,
    seed=21,
)

print("Sum of random-surfer scores:", sum(random_scores.values()))
plot_web_graph(website.graph, title="Random-surfer scores (larger node = larger score)", scores=random_scores)
plot_rank_bars(random_scores, title="PageRank estimated by random surfing")


Sum of random-surfer scores: 1.0


### 4.3 Your turn — reason about the simulation

1. Run the cell above twice with the same `seed`. Are the results identical?
2. Change the seed. What changes?
3. Increase `steps_per_walk` while keeping `repeats` fixed. Does the ranking become more stable?
4. Keep the total number of visits roughly fixed, but use more repetitions with shorter walks. Compare the results.
5. Try `alpha=0.5` and `alpha=0.95`. Which pages gain or lose score?

There is a trade-off: more visits generally reduce simulation noise, but require more computation.

In [13]:
# Change one setting at a time on the same five pages.
trials = [
    # Label, alpha, steps per walk, number of walks, seed
    ("baseline",      0.85, 12000, 12, 21),
    ("seed 22",       0.85, 12000, 12, 22),
    ("short seed 21", 0.85,  1000, 12, 21),
    ("short seed 22", 0.85,  1000, 12, 22),
    ("48 walks",      0.85,  3000, 48, 21),
    ("alpha 0.50",    0.50, 12000, 12, 21),
    ("alpha 0.95",    0.95, 12000, 12, 21),
]

walk_scores = {}
for label, alpha, steps, walks, seed in trials:
    walk_scores[label] = pagerank_random_surfer(
        website.graph, alpha=alpha, steps_per_walk=steps, repeats=walks, seed=seed
    )

print("Same scores when I repeat seed 21:", walk_scores["baseline"] == random_scores)
walk_table = pd.DataFrame.from_dict(walk_scores, orient="index")
display(walk_table.round(5))

for first, second in [("short seed 21", "short seed 22"), ("baseline", "seed 22")]:
    gap = (walk_table.loc[first] - walk_table.loc[second]).abs().sum()
    print(f"Total score difference between {first} and {second}: {gap:.5f}")


Same scores when I repeat seed 21: True


,search.com,social.com,shop.com,news.com,weather.com
baseline,0.31276,0.31295,0.16865,0.16873,0.03691
seed 22,0.31282,0.31184,0.16979,0.16865,0.03690
short seed 21,0.31242,0.31258,0.17225,0.16742,0.03533
short seed 22,0.31017,0.31025,0.17017,0.17117,0.03825
48 walks,0.31273,0.31293,0.16861,0.16875,0.03698
alpha 0.50,0.26776,0.26678,0.17728,0.17699,0.11119
alpha 0.95,0.32667,0.32648,0.16656,0.16708,0.01320


Total score difference between short seed 21 and short seed 22: 0.01333
Total score difference between baseline and seed 22: 0.00240


1. Yes. Repeating the run with seed 21 gave me exactly the same five scores.
2. With seed 22, `search.com` came just ahead of `social.com`. With seed 21, `social.com` came first. Changing the seed changes the path the surfer takes.
3. I kept 12 walks and compared seeds 21 and 22. When I went from 1,000 to 12,000 steps per walk, the sum of the score differences in absolute value fell from 0.01333 to 0.00240. The scores were more stable, but close pages could still switch places.
4. I compared 12 walks of 12,000 steps with 48 walks of 3,000 steps. Both used 144,000 visits. The page order stayed the same, and the largest score change was about 0.00007.
5. When I raised `alpha` from 0.50 to 0.95, `search.com` and `social.com` gained score. `shop.com`, `news.com` and `weather.com` lost score. The biggest drop was for `weather.com` (0.11119 to 0.01320). No other page links to it, so it gets fewer visits when surfers follow links more often.

## 5. Method 2 — PageRank by matrix and power iteration

The second method describes all transitions in one matrix. It avoids simulating individual visitors.

### 5.1 Choose a matrix convention

We fix an order for the pages. The order is important because it determines the meaning of every row and column.

We use a **column-stochastic** transition matrix $S$:

- Column $j$ represents the page we are currently on (the source page).
- Row $i$ represents the page we move to (the destination page).
- $S_{ij}$ is the probability of moving from page $j$ to page $i$ by following the link rule.

If source page $j$ has $d_j$ outgoing links, each destination receives probability $1/d_j$. If it is dangling, we replace its column with a uniform distribution over all $n$ pages. Therefore, every column of $S$ sums to 1.

For our five pages, the order is:

1. `search.com`
2. `social.com`
3. `shop.com`
4. `news.com`
5. `weather.com`

The last page has no outgoing links, so its column contains $1/5$ in every row.

### 5.2 The transition matrix for our example

With rows and columns ordered as above, the link-following matrix is:

$$
S =
\begin{bmatrix}
0 & \frac{1}{2} & 1 & 0 & \frac{1}{5} \\
\frac{1}{2} & 0 & 0 & 1 & \frac{1}{5} \\
0 & \frac{1}{2} & 0 & 0 & \frac{1}{5} \\
\frac{1}{2} & 0 & 0 & 0 & \frac{1}{5} \\
0 & 0 & 0 & 0 & \frac{1}{5}
\end{bmatrix}
$$

For example, the first column says that a visitor on `search.com` chooses `social.com` or `news.com`, each with probability $1/2$. The fifth column says that a visitor on the dangling page `weather.com` is sent uniformly to any of the five pages.

A useful check is:

$$
\sum_{i=1}^{n} S_{ij} = 1 \quad \text{for every column } j
$$

### 5.3 Teleportation and the Google matrix

The random-surfer method also allows a visitor to teleport with probability $1-\alpha$. We add this rule to the matrix.

Let $\mathbf{1}$ be an $n$-dimensional column vector of ones. The all-ones matrix is $\mathbf{1}\mathbf{1}^{\mathsf T}$. It has shape $n\times n$.

The Google matrix is:

$$
G = \alpha S + (1-\alpha)\frac{1}{n}\mathbf{1}\mathbf{1}^{\mathsf T}
$$

For $n=5$ and $\alpha=0.85$:

$$
G = 0.85S + 0.03
\begin{bmatrix}
1&1&1&1&1\\
1&1&1&1&1\\
1&1&1&1&1\\
1&1&1&1&1\\
1&1&1&1&1
\end{bmatrix}
$$

Every entry receives a small teleportation contribution. This makes it possible to reach every page, even if the original link graph is disconnected.

### 5.4 Matrix-vector multiplication

A PageRank vector stores one score per page. We use a column vector:

$$
\mathbf{p}^{(t)} =
\begin{bmatrix}
p_{\text{search}}^{(t)}\\
p_{\text{social}}^{(t)}\\
p_{\text{shop}}^{(t)}\\
p_{\text{news}}^{(t)}\\
p_{\text{weather}}^{(t)}
\end{bmatrix}
$$

One iteration is a matrix-vector product. In the five-page example, its structure is:

$$
\begin{bmatrix}
G_{11} & G_{12} & G_{13} & G_{14} & G_{15}\\
G_{21} & G_{22} & G_{23} & G_{24} & G_{25}\\
G_{31} & G_{32} & G_{33} & G_{34} & G_{35}\\
G_{41} & G_{42} & G_{43} & G_{44} & G_{45}\\
G_{51} & G_{52} & G_{53} & G_{54} & G_{55}
\end{bmatrix}
\begin{bmatrix}
p_{\text{search}}^{(t)}\\
p_{\text{social}}^{(t)}\\
p_{\text{shop}}^{(t)}\\
p_{\text{news}}^{(t)}\\
p_{\text{weather}}^{(t)}
\end{bmatrix}
=
\begin{bmatrix}
p_{\text{search}}^{(t+1)}\\
p_{\text{social}}^{(t+1)}\\
p_{\text{shop}}^{(t+1)}\\
p_{\text{news}}^{(t+1)}\\
p_{\text{weather}}^{(t+1)}
\end{bmatrix}
$$

The dimensions are:

$$
\underbrace{G}_{n\times n}
\underbrace{\mathbf{p}^{(t)}}_{n\times 1}
=
\underbrace{\mathbf{p}^{(t+1)}}_{n\times 1}
$$

In words, each destination's next score is the weighted sum of the current scores of pages that can send a visitor to it. The vector is repeatedly updated until it changes very little.

### 5.5 Build the transition and Google matrices in NumPy

NumPy arrays are indexed by integer position, so we need a dictionary from page name to matrix index. This `index` dictionary is the bridge between readable website names and numeric array positions.

We use `S[i, j]` for the probability of moving **from** page `nodes[j]` **to** page `nodes[i]`.

**Your turn — implement the matrix construction.** In the next code cell, write the complete construction of the column-stochastic link matrix `S`. For each source page, distribute its column equally across outgoing destinations; for a dangling page, fill its column with a uniform distribution. Then construct the teleportation matrix and combine it with `S` using the formula from Section 5.3. Return `(google, S, nodes)` in that order.

In [14]:
def build_google_matrix(G, alpha=0.85):
    """Return the Google matrix, link matrix, and page order for graph G."""
    if not 0 <= alpha <= 1:
        raise ValueError("alpha must be between 0 and 1")

    nodes = list(G.nodes())
    n = len(nodes)
    if n == 0:
        return np.zeros((0, 0)), np.zeros((0, 0)), nodes

    index = {node: position for position, node in enumerate(nodes)}
    S = np.zeros((n, n))

    for source in nodes:
        j = index[source]
        destinations = list(G.successors(source))

        if destinations:
            probability = 1.0 / len(destinations)
            for destination in destinations:
                i = index[destination]
                S[i, j] = probability
        else:
            S[:, j] = 1.0 / n

    teleport = np.full((n, n), 1.0 / n)
    google = alpha * S + (1 - alpha) * teleport
    return google, S, nodes

In [ ]:
google_matrix, link_matrix, matrix_nodes = build_google_matrix(website.graph, alpha=0.85)

print("Page order:", matrix_nodes)
print("Column sums of S:", link_matrix.sum(axis=0))
print("Column sums of G:", google_matrix.sum(axis=0))

print("Link-following matrix S:")
display(pd.DataFrame(link_matrix, index=matrix_nodes, columns=matrix_nodes).round(3))

print("Google matrix G:")
display(pd.DataFrame(google_matrix, index=matrix_nodes, columns=matrix_nodes).round(3))

plot_matrix_heatmap(link_matrix, matrix_nodes, title="Link-following matrix S")
plot_matrix_heatmap(google_matrix, matrix_nodes, title="Google matrix G")


Page order: ['search.com', 'social.com', 'shop.com', 'news.com', 'weather.com']
Column sums of S: [1. 1. 1. 1. 1.]
Column sums of G: [1. 1. 1. 1. 1.]
Link-following matrix S:


,search.com,social.com,shop.com,news.com,weather.com
search.com,0.0,0.5,1.0,0.0,0.2
social.com,0.5,0.0,0.0,1.0,0.2
shop.com,0.0,0.5,0.0,0.0,0.2
news.com,0.5,0.0,0.0,0.0,0.2
weather.com,0.0,0.0,0.0,0.0,0.2


Google matrix G:


,search.com,social.com,shop.com,news.com,weather.com
search.com,0.030,0.455,0.88,0.03,0.2
social.com,0.455,0.030,0.03,0.88,0.2
shop.com,0.030,0.455,0.03,0.03,0.2
news.com,0.455,0.030,0.03,0.03,0.2
weather.com,0.030,0.030,0.03,0.03,0.2


### 5.6 Power iteration

Start with a uniform score vector: every page initially receives $1/n$.

Then repeat:

$$
\mathbf{p}^{(t+1)} = G\mathbf{p}^{(t)}.
$$

We stop when the L1 change is smaller than a tolerance:

$$
\left\|\mathbf{p}^{(t+1)}-\mathbf{p}^{(t)}\right\|_1
= \sum_{i=1}^{n}\left|p_i^{(t+1)}-p_i^{(t)}\right|
< \text{tol}.
$$

The tolerance is a numerical stopping rule. `max_iter` is a safety limit so that the loop cannot continue forever if a problem occurs.

**Your turn — implement the complete power-iteration procedure.** The next code cell supplies the Google matrix, an initial uniform NumPy vector, and a history list containing that initial vector. Write the iteration loop, record each updated vector, use the L1 norm as the stopping criterion, and respect `max_iter`. Finally, create the page-name-to-score dictionary and return it with the history list.

In [16]:
def pagerank_power_iteration(G, alpha=0.85, tol=1e-10, max_iter=1000):
    """Calculate PageRank by repeatedly multiplying by the Google matrix."""
    if tol <= 0:
        raise ValueError("tol must be positive")
    if max_iter <= 0:
        raise ValueError("max_iter must be positive")

    google, _, nodes = build_google_matrix(G, alpha=alpha)
    n = len(nodes)
    if n == 0:
        return {}, []

    scores = np.full(n, 1.0 / n)
    history = [scores.copy()]

    for _ in range(max_iter):
        new_scores = google @ scores
        change = np.sum(np.abs(new_scores - scores))

        scores = new_scores
        history.append(scores.copy())

        if change < tol:
            break

    scores_dict = {node: float(scores[i]) for i, node in enumerate(nodes)}
    return scores_dict, history

In [ ]:
power_scores, power_history = pagerank_power_iteration(
    website.graph,
    alpha=0.85,
    tol=1e-12,
    max_iter=1000,
)

print("Power iteration updates:", len(power_history) - 1)
print("Sum of power-iteration scores:", sum(power_scores.values()))
plot_web_graph(website.graph, title="Power-iteration scores (larger node = larger score)", scores=power_scores)
plot_rank_bars(power_scores, title="PageRank calculated by power iteration")


Power iteration updates: 32
Sum of power-iteration scores: 1.0000000000000013


In [ ]:
# Plot how much the score vector changes at each iteration.
changes = [
    np.linalg.norm(power_history[i + 1] - power_history[i], ord=1)
    for i in range(len(power_history) - 1)
]

fig = go.Figure(go.Scatter(
    x=list(range(1, len(changes) + 1)),
    y=changes,
    mode="lines+markers",
    hovertemplate="Iteration %{x}<br>L1 change: %{y:.3e}<extra></extra>",
))
fig.update_layout(
    title="Power-iteration convergence",
    xaxis_title="Iteration",
    yaxis_title="L1 change between consecutive score vectors",
    yaxis_type="log",
    height=400,
)
fig.show()


### 5.7 Compare the two methods

The methods approximate the same PageRank model, but they use different numerical procedures:

- The random surfer estimates scores from sampled visits. Its answer has sampling noise.
- Power iteration repeatedly applies the transition matrix. Its answer is controlled by a numerical tolerance and maximum iteration count.

The total of each score vector should be 1. The two rankings should usually be similar, although finite random walks will not match the matrix result exactly.

In [19]:
comparison = pd.DataFrame({
    "random surfer": pd.Series(random_scores),
    "power iteration": pd.Series(power_scores),
})
comparison["absolute difference"] = (comparison["random surfer"] - comparison["power iteration"]).abs()
comparison = comparison.sort_values("power iteration", ascending=False)
display(comparison.round(6))

l1_error = sum(abs(random_scores[node] - power_scores[node]) for node in website.graph.nodes())
print("L1 difference between the two score vectors:", round(l1_error, 6))


,random surfer,power iteration,absolute difference
search.com,0.312764,0.312830,0.000066
social.com,0.312951,0.312830,0.000121
shop.com,0.168646,0.169097,0.000452
news.com,0.168729,0.169097,0.000368
weather.com,0.036910,0.036145,0.000765


L1 difference between the two score vectors: 0.001773


**Your turn — interpret the comparison**

- Are the pages in the same order in both methods?
- Which page has the largest absolute score difference?
- What happens to the difference when you increase the number of random-surfer steps and repetitions?
- Why is it reasonable that the two result dictionaries are not exactly equal?

In [20]:
# Repeat the five-page experiment with twice as many steps and walks.
larger_sample = pagerank_random_surfer(website.graph, alpha=0.85,
                                       steps_per_walk=24000, repeats=24, seed=21)
larger_l1 = sum(abs(larger_sample[page] - power_scores[page]) for page in power_scores)
print(f"L1 error with 144,000 visits: {l1_error:.6f}")
print(f"L1 error with 576,000 visits: {larger_l1:.6f}")


L1 error with 144,000 visits: 0.001773
L1 error with 576,000 visits: 0.000322


1. I got the same top pair with both methods. In the matrix result, `search.com` and `social.com` tie at 0.312830. The walk puts `social.com` slightly ahead. The middle pair also ties in the matrix result, and the walk puts `news.com` ahead of `shop.com`. Both methods put `weather.com` last.
2. The largest gap is for `weather.com`. I got 0.036910 from the walk and 0.036145 from the matrix, a difference of 0.000765.
3. When I doubled both the steps and the walks, the total went from 144,000 to 576,000 visits. The sum of the absolute differences fell from 0.001773 to 0.000322. With more visits I expect less error on average.
4. The walk counts where the surfer happened to go. The matrix uses the probability of every possible move. A finite walk can visit a page a little more or less often than its probability predicts.

## 6. Testing useful properties

A good algorithm is not only one that produces a plausible answer. We should check properties that must be true.

For this PageRank model:

1. Each score should be non-negative.
2. The scores should sum to 1.
3. Every column of the Google matrix should sum to 1.
4. A graph with no nodes should return an empty score dictionary.

These are small tests, but they catch many implementation mistakes.

In [21]:
assert all(score >= 0 for score in power_scores.values())
assert abs(sum(power_scores.values()) - 1.0) < 1e-9
assert np.allclose(google_matrix.sum(axis=0), np.ones(len(matrix_nodes)))
assert pagerank_power_iteration(nx.DiGraph())[0] == {}

print("All basic PageRank checks passed.")


All basic PageRank checks passed.


## 7. Random graph generation: make test networks of a chosen size

We now want to study larger graphs. Before writing the generator, recall two ways to interpret randomness:

### 7.1 Fixed number of edges versus independent edge probability

There are two common models:

**Fixed-size model:** choose exactly $m$ directed edges from the possible source-destination pairs. If self-links are excluded, there are $n(n-1)$ possible pairs. Our benchmark generator will use this model because you can request an exact number of nodes and edges.

**Independent-edge model:** examine each possible directed pair and add the edge with probability $p$. Each edge decision is a Bernoulli trial. The number of edges varies from graph to graph. Its expected value is:

$$
\mathbb{E}[M] = p\,n(n-1)
$$

when self-links are excluded. This model is useful when you want a probability-controlled random graph, but it does not guarantee an exact edge count.

### 7.2 Sampling without replacement

For a fixed-size graph, we build all valid directed pairs and call `rng.sample(candidates, m)`. Sampling is **without replacement**, so the same directed edge cannot be selected twice. The generator below excludes self-links (`source == target`).

A random graph is a test input, not a realistic model of the entire web. Real websites have communities, hubs, and other structure that a uniform random graph does not capture.

**Your turn — implement independent-edge sampling.** In the next code cell, complete `sample_edge_count(n, p, seed)`. Examine every ordered pair of distinct pages, include each possible directed edge independently with probability `p`, and return the resulting edge count. Use the supplied random-number generator so that a seed gives reproducible results.

In [22]:
def sample_edge_count(n, p, seed=7):
    """Sample each possible directed edge independently and count the selected edges."""
    if n < 0:
        raise ValueError("n must be non-negative")
    if not 0 <= p <= 1:
        raise ValueError("p must be between 0 and 1")

    rng = random.Random(seed)
    count = 0
    for source in range(n):
        for target in range(n):
            if source != target and rng.random() < p:
                count += 1
    return count

# Four pages have 4 × 3 possible directed links.
assert sample_edge_count(4, 0) == 0
assert sample_edge_count(4, 1) == 12
print("Links sampled with p=0.5:", sample_edge_count(4, 0.5, seed=7))

Links sampled with p=0.5: 9


### 7.3 Website names for generated graphs

The first few generated nodes use readable website-style names. For larger graphs, we use names such as `site-021.com`, `site-022.com`, and so on. This avoids uninformative single-letter node labels while keeping names unique.

**Your turn — implement the fixed-size graph generator.** In the next code cell, complete `make_random_web_graph(n, m, seed)`. Generate all valid directed pairs without self-links, sample exactly `m` distinct pairs without replacement, and return a NetworkX directed graph containing all `n` named pages and the sampled edges. The function's input validation and page-name helper are supplied.

In [23]:
def make_website_names(n):
    """Return n short, unique website-style names."""
    common_names = [
        "search.com", "social.com", "shop.com", "news.com", "video.com",
        "travel.com", "music.com", "forum.com", "maps.com", "bank.com",
        "weather.com", "books.com", "games.com", "photo.com", "food.com",
        "sports.com", "learn.com", "cloud.com", "mail.com", "design.com",
    ]
    if n <= len(common_names):
        return common_names[:n]
    extra_names = [f"site-{i:03d}.com" for i in range(len(common_names) + 1, n + 1)]
    return common_names + extra_names


def make_random_web_graph(n, m, seed=7):
    """Create a directed graph with exactly n nodes and m non-self edges."""
    if n < 0:
        raise ValueError("n must be non-negative")
    if m < 0 or m > n * (n - 1):
        raise ValueError("m must be between 0 and n * (n - 1)")

    nodes = make_website_names(n)
    rng = random.Random(seed)

    candidates = []
    for source in nodes:
        for target in nodes:
            if source != target:
                candidates.append((source, target))

    edges = rng.sample(candidates, m)

    graph = nx.DiGraph()
    graph.add_nodes_from(nodes)
    graph.add_edges_from(edges)
    return graph

In [ ]:
random_example = make_random_web_graph(n=12, m=24, seed=9)
print("Nodes:", random_example.number_of_nodes())
print("Edges:", random_example.number_of_edges())
plot_web_graph(random_example, title="A generated directed web graph")


Nodes: 12
Edges: 24


### 7.4 Check the generator

For each generated graph, we should be able to verify the requested size exactly. We should also check that no self-links were created.

In [25]:
check_graph = make_random_web_graph(n=30, m=90, seed=42)
assert check_graph.number_of_nodes() == 30
assert check_graph.number_of_edges() == 90
assert all(source != target for source, target in check_graph.edges())
print("Generator checks passed.")


Generator checks passed.


## 8. Performance analysis: how does running time grow?

We now compare the running time of the two methods on generated graphs.

### 8.1 What is being measured?

- **Random surfer:** we set the length of each walk proportional to $n$, so the total number of simulated visits grows with the number of pages. The number of repetitions is fixed. This is a chosen simulation budget; a larger budget usually improves the estimate but takes longer.
- **Power iteration:** the implementation builds a dense $n\times n$ Google matrix and performs repeated dense matrix-vector products.

The main costs are approximately:

$$
T_{\text{random}} = O(RL),
$$

where $R$ is the number of independent walks and $L$ is the number of steps per walk, and

$$
T_{\text{dense power}} = O(n^2 + I n^2),
$$

where $I$ is the number of power iterations. The dense matrix also needs $O(n^2)$ memory.

These are broad complexity descriptions, not exact timing predictions. Python overhead, NumPy's implementation, graph structure, and the number of iterations also matter.

A sparse matrix implementation can reduce the power-iteration cost to about $O(I(n+m))$ for a sparse graph. We keep the dense NumPy version here because it makes the matrix and vector operations easy to inspect.

### 8.2 A reusable timing helper

We use `time.perf_counter()` for elapsed time. We repeat each measurement several times and report the median, which is less sensitive to one unusually slow or fast run.

The time for generating the graph is **not** included. We measure the PageRank algorithms on a graph that has already been created.

In [26]:
def median_runtime(function, timing_repeats=3):
    """Run function several times and return the median elapsed time in seconds."""
    elapsed = []
    for trial in range(timing_repeats):
        start = time.perf_counter()
        function(trial)
        elapsed.append(time.perf_counter() - start)
    return median(elapsed)


def benchmark_one_graph(G, steps_per_node=150, walk_repeats=3, timing_repeats=3, seed=100):
    """Time both PageRank methods on one graph and return a result dictionary."""
    n = G.number_of_nodes()
    steps_per_walk = max(1, steps_per_node * n)

    random_time = median_runtime(
        lambda trial: pagerank_random_surfer(
            G,
            alpha=0.85,
            steps_per_walk=steps_per_walk,
            repeats=walk_repeats,
            seed=seed + trial,
        ),
        timing_repeats=timing_repeats,
    )

    power_time = median_runtime(
        lambda trial: pagerank_power_iteration(
            G,
            alpha=0.85,
            tol=1e-8,
            max_iter=1000,
        ),
        timing_repeats=timing_repeats,
    )

    # Keep one iteration count as useful context, measured outside the timed calls.
    _, history = pagerank_power_iteration(G, alpha=0.85, tol=1e-8, max_iter=1000)

    return {
        "nodes": n,
        "edges": G.number_of_edges(),
        "random_surfer_seconds": random_time,
        "power_iteration_seconds": power_time,
        "power_iterations": max(0, len(history) - 1),
    }


### 8.3 Experiment A — increase the number of pages

We keep the average number of outgoing links roughly fixed at four by setting $m=4n$. The number of pages increases, and the number of edges increases with it.

For a fair interpretation, remember that the random-surfer sample budget also increases with $n$ in this experiment: each walk has `steps_per_node * n` steps.

In [27]:
node_sizes = list(range(10,200,20))
node_results = []

for n in node_sizes:
    m = min(4 * n, n * (n - 1))
    G = make_random_web_graph(n=n, m=m, seed=1000 + n)
    result = benchmark_one_graph(
        G,
        steps_per_node=150,
        walk_repeats=3,
        timing_repeats=9,
        seed=1000 + n,
    )
    node_results.append(result)

node_benchmark = pd.DataFrame(node_results)
display(node_benchmark.round(5))


,nodes,edges,random_surfer_seconds,power_iteration_seconds,power_iterations
0,10,40,0.00119,0.00007,20
1,30,120,0.00342,0.00012,22
2,50,200,0.00598,0.00027,24
3,70,280,0.00926,0.00037,26
4,90,360,0.01894,0.00052,23
5,110,440,0.02365,0.00389,27
6,130,520,0.02793,0.00395,27
7,150,600,0.03407,0.00435,26
8,170,680,0.04038,0.00401,25
9,190,760,0.04582,0.00353,24


In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=node_benchmark["nodes"], y=node_benchmark["random_surfer_seconds"],
    mode="lines+markers", name="Random surfer",
    hovertemplate="Nodes: %{x}<br>Seconds: %{y:.5f}<extra></extra>",
))
fig.add_trace(go.Scatter(
    x=node_benchmark["nodes"], y=node_benchmark["power_iteration_seconds"],
    mode="lines+markers", name="Dense power iteration",
    hovertemplate="Nodes: %{x}<br>Seconds: %{y:.5f}<extra></extra>",
))
fig.update_layout(
    title="Runtime as the number of pages increases (about 4 links per page)",
    xaxis_title="Number of pages (nodes)",
    yaxis_title="Median runtime (seconds)",
    height=450,
)
fig.show()


### 8.4 Experiment B — increase the number of links

Now we keep the number of pages fixed and vary the number of directed edges. This helps separate the effect of $n$ from the effect of $m$.

The dense matrix has $n^2$ entries regardless of how many links are present. As a result, its matrix-vector multiplication cost is strongly driven by $n$, not directly by $m$. The random surfer stores each node's outgoing-neighbour list and chooses from it at each step; its runtime can also be affected by Python and graph structure, but the number of simulated steps is the main control in this experiment.

In [29]:
fixed_n = 100
edge_counts = [100, 300, 700, 1500, 3000, 6000]
edge_results = []

for m in edge_counts:
    G = make_random_web_graph(n=fixed_n, m=m, seed=3000 + m)
    result = benchmark_one_graph(
        G,
        steps_per_node=150,
        walk_repeats=3,
        timing_repeats=3,
        seed=4000 + m,
    )
    edge_results.append(result)

edge_benchmark = pd.DataFrame(edge_results)
display(edge_benchmark.round(5))


,nodes,edges,random_surfer_seconds,power_iteration_seconds,power_iterations
0,100,100,0.02526,0.00391,23
1,100,300,0.02608,0.00528,32
2,100,700,0.01589,0.00246,18
3,100,1500,0.02506,0.00262,12
4,100,3000,0.02568,0.00175,10
5,100,6000,0.02390,0.00174,7


In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=edge_benchmark["edges"], y=edge_benchmark["random_surfer_seconds"],
    mode="lines+markers", name="Random surfer",
    hovertemplate="Edges: %{x}<br>Seconds: %{y:.5f}<extra></extra>",
))
fig.add_trace(go.Scatter(
    x=edge_benchmark["edges"], y=edge_benchmark["power_iteration_seconds"],
    mode="lines+markers", name="Dense power iteration",
    hovertemplate="Edges: %{x}<br>Seconds: %{y:.5f}<extra></extra>",
))
fig.update_layout(
    title=f"Runtime as the number of links increases (fixed at {fixed_n} pages)",
    xaxis_title="Number of directed links (edges)",
    yaxis_title="Median runtime (seconds)",
    height=450,
)
fig.show()


### 8.5 Discuss the results

Use the plots and tables to answer:

1. Which method is faster for the graph sizes you tested?
2. Does the runtime grow in the same way when the number of pages increases?
3. When the number of pages is fixed, does adding more links have a large effect on dense power iteration?
4. What happens if you double `steps_per_node`? What happens if you double `walk_repeats`?
5. Which method uses a dense $n\times n$ matrix? What does this imply for memory use when $n$ becomes large?
6. How could a sparse matrix representation change the matrix method?

Your measured values will depend on your computer, Python version, and installed numerical libraries. Focus on the trend and explain the experimental settings; do not assume one machine's exact timings are universal.

In [31]:
# Use the same graph and seeds for all three budgets.
budget_graph = make_random_web_graph(n=100, m=300, seed=3300)
budget_rows = []
for steps, walks in [(150, 3), (300, 3), (150, 6)]:
    result = benchmark_one_graph(budget_graph, steps_per_node=steps, walk_repeats=walks,
                                 timing_repeats=7, seed=4300)
    budget_rows.append({
        "steps per node": steps, "walks": walks, "visits": steps * 100 * walks,
        "random surfer (ms)": 1000 * result["random_surfer_seconds"],
        "power iteration (ms)": 1000 * result["power_iteration_seconds"],
    })
display(pd.DataFrame(budget_rows).round(3))


,steps per node,walks,visits,random surfer (ms),power iteration (ms)
0,150,3,45000,23.216,5.198
1,300,3,90000,47.845,4.735
2,150,6,90000,48.759,6.212


1. Power iteration was faster for all 16 graphs I tested. At 190 pages and 760 links, it took 3.53 ms, while the walk took 45.82 ms. These are medians with the settings above.
2. The walk took longer as I added pages. I gave it 450 visits per page, so 10 pages meant 4,500 visits and 190 pages meant 85,500. Power iteration took 0.07 ms at 10 pages and 3.53 ms at 190. Each iteration works on a whole square matrix, and the number of iterations also varies from graph to graph.
3. Adding more links did not give a steady rise in time. At 100 pages, the matrix stayed the same size. The 300-link graph took 32 iterations (5.28 ms), and the 6,000-link graph took 7 (1.74 ms).
4. The walk took 23.22 ms for 45,000 visits. Doubling the steps per walk gave 47.85 ms. Doubling the number of walks instead gave 48.76 ms. Each change doubled the visits to 90,000. Power iteration uses neither of these settings.
5. Power iteration stores a dense matrix with one row and one column per page. Doubling the number of pages takes four times the matrix memory.
6. Most entries in the link matrix are zero when each page has only a few links. I can store just the links and their weights, then add the random-jump probability and the share from dead ends separately. This takes less memory and skips the zero entries during multiplication.

## 9. Optional extensions

These are suitable follow-up tasks for students who finish early.

### Extension A — compare with NetworkX's built-in PageRank

NetworkX includes a PageRank implementation. Use it as an independent reference on the small example graph, then compare its output with power iteration. Check the NetworkX documentation for the exact meaning of its `alpha`, `tol`, and dangling-node options.

```python
reference_scores = nx.pagerank(website.graph, alpha=0.85, tol=1e-12)
```

In [32]:
reference = nx.pagerank(website.graph, alpha=0.85, tol=1e-12, max_iter=1000)
reference_table = pd.DataFrame({
    "my power iteration": pd.Series(power_scores),
    "NetworkX": pd.Series(reference),
})
display(reference_table.round(6))
max_gap = (reference_table["my power iteration"] - reference_table["NetworkX"]).abs().max()
print(f"Largest difference: {max_gap:.2e}")

,my power iteration,NetworkX
search.com,0.312830,0.312830
social.com,0.312830,0.312830
shop.com,0.169097,0.169097
news.com,0.169097,0.169097
weather.com,0.036145,0.036145


Largest difference: 2.48e-13


The five scores agree to six decimal places. Both runs use an 85% chance of following a link, and both spread a dangling page's score equally across the five pages. My loop stops when the change is below `tol`. NetworkX uses `len(G) * tol`, which explains why the last few digits can differ. [NetworkX PageRank documentation](https://networkx.org/documentation/stable/reference/algorithms/generated/networkx.algorithms.link_analysis.pagerank_alg.pagerank.html)

### Extension B — change the graph

Add a page called `blog.com`, then add links from `blog.com` to `news.com` and `social.com`. Re-run both algorithms. Explain why the scores move.

In [33]:
blog_graph = website.graph.copy()
blog_graph.add_edges_from([("blog.com", "news.com"), ("blog.com", "social.com")])

blog_walk = pagerank_random_surfer(blog_graph, steps_per_walk=12000, repeats=12, seed=21)
blog_power, _ = pagerank_power_iteration(blog_graph, tol=1e-12)
blog_table = pd.DataFrame({
    "original (power)": pd.Series(power_scores),
    "with blog (power)": pd.Series(blog_power),
    "with blog (walk)": pd.Series(blog_walk),
})
display(blog_table.round(5))

,original (power),with blog (power),with blog (walk)
blog.com,NaN,0.02913,0.02947
news.com,0.16910,0.16868,0.16884
search.com,0.31283,0.29924,0.29885
shop.com,0.16910,0.16175,0.16106
social.com,0.31283,0.31207,0.31227
weather.com,0.03614,0.02913,0.02952


The two new links break the old ties. In the matrix result, `social.com` (0.3121) now beats `search.com` (0.2992), and `news.com` (0.1687) beats `shop.com` (0.1618). The walk gives the same order for these four pages. Social and news both lose a little compared with their original scores because six pages now share the total. The direct links from the blog help them lose less than their former partners. Blog and weather tie at 0.0291 because neither has any incoming links.

### Extension C — sparse matrix implementation

For large sparse graphs, investigate `scipy.sparse` and compare sparse matrix-vector multiplication with the dense NumPy approach. Keep the same column-stochastic convention so that the comparison is meaningful.

In [34]:
from scipy.sparse import csr_matrix
from timeit import repeat

sparse_graph = make_random_web_graph(400, 1200, seed=12)
dense_google, _, page_order = build_google_matrix(sparse_graph, alpha=0.85)
n_pages = len(page_order)
index = {page: i for i, page in enumerate(page_order)}

rows, cols, weights = [], [], []
dangling = []
for source in page_order:
    targets = list(sparse_graph.successors(source))
    if not targets:
        dangling.append(index[source])
    for target in targets:
        rows.append(index[target])
        cols.append(index[source])
        weights.append(1 / len(targets))
link_csr = csr_matrix((weights, (rows, cols)), shape=(n_pages, n_pages))

def sparse_step(scores):
    shared = (0.85 * scores[dangling].sum() + 0.15) / n_pages
    return 0.85 * (link_csr @ scores) + shared

# Start with unequal scores so the test checks which page gets each contribution.
start_scores = np.arange(1, n_pages + 1, dtype=float)
start_scores /= start_scores.sum()
dense_result = dense_google @ start_scores
sparse_result = sparse_step(start_scores)
assert np.allclose(sparse_result, dense_result)
print(f"Largest difference after one step: {np.max(np.abs(sparse_result - dense_result)):.2e}")

# Build the two matrices once, then time the same step 20 times per trial.
for label, step in [("dense", lambda: dense_google @ start_scores),
                    ("sparse", lambda: sparse_step(start_scores))]:
    times = repeat(step, number=20, repeat=3)
    print(f"{label}: {1000 * median(times) / 20:.4f} ms per step")
csr_bytes = link_csr.data.nbytes + link_csr.indices.nbytes + link_csr.indptr.nbytes
print(f"Dense matrix: {dense_google.nbytes:,} bytes. Sparse link matrix: {csr_bytes:,} bytes.")

Largest difference after one step: 1.73e-18
dense: 0.1725 ms per step
sparse: 0.0190 ms per step
Dense matrix: 1,280,000 bytes. Sparse link matrix: 16,004 bytes.


The two updates gave the same scores to rounding. On 400 pages with 1,200 links, one dense update took 0.1725 ms and one sparse update took 0.0190 ms in this run. The dense matrix stored 160,000 entries. The sparse version stored the 1,200 links and handled random jumps separately. The timing leaves out the cost of building the matrices.

### Extension D — test special cases

Try an empty graph, a graph with one page, a graph with no links, a graph with one-way links only, and a graph with several dangling pages. Explain how teleportation keeps the process defined.

In [35]:
cases = {
    "empty": ([], []),
    "one page": (["A"], []),
    "no links": (["A", "B", "C"], []),
    "one-way chain": (["A", "B", "C"], [("A", "B"), ("B", "C")]),
    "several dangling": (["A", "B", "C", "D"], [("A", "B"), ("A", "C")]),
}
for name, (nodes, edges) in cases.items():
    graph = nx.DiGraph()
    graph.add_nodes_from(nodes)
    graph.add_edges_from(edges)
    matrix_scores, _ = pagerank_power_iteration(graph)
    walk_scores = pagerank_random_surfer(graph, steps_per_walk=5000, repeats=4, seed=21)
    if nodes:
        assert np.isclose(sum(matrix_scores.values()), 1)
        assert np.isclose(sum(walk_scores.values()), 1)
    else:
        assert matrix_scores == walk_scores == {}
    print(name)
    print("  matrix:", {p: round(v, 3) for p, v in matrix_scores.items()})
    print("  walk:  ", {p: round(v, 3) for p, v in walk_scores.items()})

empty
  matrix: {}
  walk:   {}
one page
  matrix: {'A': 1.0}
  walk:   {'A': 1.0}
no links
  matrix: {'A': 0.333, 'B': 0.333, 'C': 0.333}
  walk:   {'A': 0.331, 'B': 0.339, 'C': 0.331}
one-way chain
  matrix: {'A': 0.184, 'B': 0.341, 'C': 0.474}
  walk:   {'A': 0.187, 'B': 0.344, 'C': 0.469}
several dangling
  matrix: {'A': 0.206, 'B': 0.294, 'C': 0.294, 'D': 0.206}
  walk:   {'A': 0.206, 'B': 0.291, 'C': 0.297, 'D': 0.207}


The empty graph gives an empty result, and the single page gets 1. With three pages and no links, each matrix score is 1/3. In the one-way chain, C comes first (0.474), followed by B (0.341) and A (0.184). In the last graph, B and C each get 0.294 because A points to both equally. At a page with no outgoing links, the next move is uniform across all pages. At a page with links, the 15% chance of a random jump still lets the surfer reach any page.

## 10. Final checklist

Before submitting your work, make sure you can explain:

- What a directed edge means and how a dangling page is handled.
- Why a list of tuples is a convenient representation for links.
- When a set or dictionary is useful.
- What the `WebsiteNetwork` class stores.
- How the random surfer chooses between following a link and teleporting.
- Why repeating and averaging simulations makes the random estimate more stable.
- What each row and column of the transition matrix represents.
- Why the Google matrix is column-stochastic.
- What the matrix-vector product $G\mathbf{p}^{(t)}$ calculates.
- How the stopping tolerance is used in power iteration.
- Why the dense matrix implementation scales differently from a random-walk simulation.
- Why benchmark results need to state the graph size and simulation settings.

**Main idea:** both PageRank methods model the same movement between pages. One estimates the long-run visit frequencies by simulation; the other calculates the same distribution through repeated matrix-vector multiplication.

**My notes**

- A link is an ordered pair. `("search.com", "news.com")` tells me which page I leave and which I reach. A set keeps one copy of each name. A dictionary lets me look up the links from a given page. `WebsiteNetwork` keeps the pages and links together in a directed graph.
- At a page with links, my surfer follows one with probability 0.85. Otherwise it jumps to a random page. At a dead end it always jumps. Averaging several walks makes one unusual path count less.
- Each matrix column describes one starting page, and each row is a possible destination. All the choices in one column add up to 1. Multiplying the matrix by the current scores gives the scores after one more move. I stop when the sum of the absolute changes falls below the tolerance.
- I need the page and link counts to interpret a timing. The dense method processes a whole square matrix at each step. For the walk, doubling the number of visits roughly doubled my time, so I also report how many steps and walks I used.